# 인간 vs AI 생성 초록 비교 분석 (Data Analysis)

이 노트북은 `proposal.txt`와 `plan.md`에 명시된 분석 계획을 실행하기 위한 기본 코드 템플릿입니다.

## 0. 라이브러리 설치 및 임포트
필요한 환경에 따라 아래 라이브러리 설치가 필요할 수 있습니다.

In [ ]:
# 필수 라이브러리 설치 (필요시 주석 해제하여 실행)
# !pip install pandas numpy matplotlib seaborn scipy transformers torch sentence-transformers konlpy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# 한글 폰트 설정 (Windows 기준)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

import warnings
warnings.filterwarnings('ignore')

## 1. 데이터 로드 및 전처리
`dbpia_computer_science.csv` 데이터를 불러오고 결측치를 확인합니다.

In [ ]:
# 데이터 로드
df = pd.read_csv('dbpia_computer_science.csv')

# 데이터 구조 확인
display(df.head())
print(df.info())

# 결측치 확인 및 제거
df = df.dropna(subset=['abstract', 'fake_abstract']).reset_index(drop=True)
print(f"결측치 제거 후 데이터 개수: {len(df)}")

## 2. 지표 측정 및 특성 추출
### 2.1. 구조적 무작위성 (Perplexity 측정)
`transformers`의 `KoGPT2`를 활용하여 각 텍스트의 Perplexity를 계산합니다. (전체 데이터 실행 시 시간이 소요될 수 있습니다.)

In [ ]:
import torch
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast

# KoGPT2 모델 및 토크나이저 로드
tokenizer_gpt = PreTrainedTokenizerFast.from_pretrained("skt/kogpt2-base-v2", bos_token='</s>', eos_token='</s>', unk_token='<unk>', pad_token='<pad>', mask_token='<mask>')
model_gpt = GPT2LMHeadModel.from_pretrained("skt/kogpt2-base-v2")
model_gpt.eval()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_gpt.to(device)

def calculate_perplexity(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return np.nan
    
    encodings = tokenizer_gpt(text, return_tensors='pt').to(device)
    max_length = model_gpt.config.n_positions
    stride = 512
    
    nlls = []
    for i in range(0, encodings.input_ids.size(1), stride):
        begin_loc = max(i + stride - max_length, 0)
        end_loc = min(i + stride, encodings.input_ids.size(1))
        trg_len = end_loc - i
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100
        
        with torch.no_grad():
            outputs = model_gpt(input_ids, labels=target_ids)
            neg_log_likelihood = outputs.loss * trg_len
        nlls.append(neg_log_likelihood)
        
    if not nlls:
        return np.nan
        
    ppl = torch.exp(torch.stack(nlls).sum() / end_loc)
    return ppl.item()

# 예시 코드 (실행 시간 단축을 위해 소량의 샘플로 먼저 테스트하는 것을 권장합니다.)
# sample_df = df.head(10).copy()
# sample_df['ppl_human'] = sample_df['abstract'].apply(calculate_perplexity)
# sample_df['ppl_ai'] = sample_df['fake_abstract'].apply(calculate_perplexity)

### 2.2. 언어적 복잡도 (Linguistic Complexity 측정)
문장 길이 및 어휘 다양성(TTR)을 계산합니다.

In [ ]:
def calculate_complexity(text):
    if not isinstance(text, str):
        return np.nan, np.nan
    
    # 1. 길이 계산 (글자 수 기준)
    length = len(text)
    
    # 2. 어휘 다양성 (Type-Token Ratio, 단순 띄어쓰기 기준)
    tokens = text.split()
    if len(tokens) == 0:
        return length, 0
    
    types = set(tokens)
    ttr = len(types) / len(tokens)
    
    return length, ttr

# 데이터 적용
df[['len_human', 'ttr_human']] = df['abstract'].apply(lambda x: pd.Series(calculate_complexity(x)))
df[['len_ai', 'ttr_ai']] = df['fake_abstract'].apply(lambda x: pd.Series(calculate_complexity(x)))

### 2.3. 초록 간 유사도 (Cosine Similarity 측정)
Sentence BERT 모델을 활용하여 텍스트 임베딩 후 코사인 유사도를 구합니다.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 한국어 문장 임베딩 모델 로드
embedder = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')

# 임베딩 벡터 추출 (전체 데이터에 대해 실행 시 시간이 소요됩니다.)
# human_embeddings = embedder.encode(df['abstract'].tolist(), convert_to_tensor=False)
# ai_embeddings = embedder.encode(df['fake_abstract'].tolist(), convert_to_tensor=False)

# Human과 그것을 모방한 AI 초록 간의 유사도(1:1 매칭)
# similarities = [cosine_similarity([h], [a])[0][0] for h, a in zip(human_embeddings, ai_embeddings)]
# df['sim_human_ai'] = similarities

## 3. 통계 분석 및 가설 검증
수집된 지표를 바탕으로 독립표본 t-검정 등을 수행합니다.

In [ ]:
# 예시: 어휘 다양성(TTR)에 대한 독립표본 t-검정
t_stat, p_val = stats.ttest_ind(df['ttr_human'].dropna(), df['ttr_ai'].dropna())

print(f"어휘 다양성(TTR) t-검정 결과: t-statistic = {t_stat:.4f}, p-value = {p_val:.4e}")
if p_val < 0.05:
    print("유의수준 0.05 하에서 인간과 AI 초록 간에 유의미한 차이가 존재합니다.")
else:
    print("인간과 AI 초록 간에 유의미한 차이가 발견되지 않았습니다.")

## 4. 데이터 시각화
추출된 지표들의 분포를 시각화하여 확인합니다.

In [ ]:
# 시각화를 위한 데이터 재구조화 (TTR 지표 예시)
plot_data = pd.DataFrame({
    'TTR': pd.concat([df['ttr_human'], df['ttr_ai']]),
    'Group': ['Human']*len(df) + ['AI']*len(df)
})

plt.figure(figsize=(8, 6))
sns.boxplot(x='Group', y='TTR', data=plot_data, palette='Set2')
plt.title('인간 vs AI 생성 초록의 어휘 다양성(TTR) 비교')
plt.ylabel('Type-Token Ratio')
plt.xlabel('작성 주체')
plt.show()